# 05 - Feature Engineering

Build the final feature matrix combining NLP, technical indicators, and market context.


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import json
from pathlib import Path

TICKERS = ['NVDA', 'AMD', 'AAPL', 'TSLA', 'MSFT']
PROCESSED_DIR = Path('../data/processed')

In [ ]:
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast).mean()
    ema_slow = series.ewm(span=slow).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal).mean()
    return macd_line - signal_line  # histogram

def compute_volume_ratio(volume, window=20):
    return volume / volume.rolling(window).mean()

In [ ]:
for ticker in TICKERS:
    df = yf.download(ticker, period='2y', interval='1d', auto_adjust=True)
    df.columns = [c.lower() for c in df.columns]
    
    # Technical indicators
    df['rsi_14'] = compute_rsi(df['close'])
    df['macd_signal'] = compute_macd(df['close'])
    df['volume_ratio'] = compute_volume_ratio(df['volume'])
    df['price_momentum_5d'] = df['close'].pct_change(5) * 100
    df['price_return_1d'] = df['close'].pct_change() * 100
    
    # Direction target: +0.5% threshold
    df['direction_label'] = df['price_return_1d'].shift(-1).apply(
        lambda x: 2 if x > 0.5 else (0 if x < -0.5 else 1)
    )
    
    df.dropna(inplace=True)
    out_path = PROCESSED_DIR / f'{ticker}_final_dataset.csv'
    df.to_csv(out_path)
    print(f'{ticker}: {len(df)} rows saved to {out_path}')